In [1]:
# packages used for creating and geo-locating the graph
import networkx as nx
import shapely

# packages needed for inspecting the output
import pandas as pd
import geopandas as gpd
import opentnsim.fis as fis

# packages needed for plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# plot libraries
import folium

import numpy as np

In [2]:
# load the processed version from the Fairway Information System graph provided by Rijkswaterstaat
# For FIS version 0.3 
FG_FIS = fis.load_network(version="0.3")

# Trying out for EuRIS
FG_EuRIS = fis.load_network(network="euris", version="0.1")

In [3]:
# Relevant polygon
poly_coords = [
    (3.3215331, 52.1076298),
    (3.2318116, 51.1184662),
    (5.0115969, 51.0782204),
    (5.0427246, 52.0789434),
    (3.3215331, 52.1076298)
]

polygon = shapely.Polygon(poly_coords)


In [4]:
# Printing the data per edge to see what the names are. Check if there is water depth info
for u, v, data in FG_FIS.edges(data=True):
    print (u, "->", v)
    print(data)
    break

8861095 -> 8864054
{'GeoType': 'section', 'Name': 'Vaarwegvak van 0 tot 2 - H', 'Length': 2.346, 'GeneralDepth': nan, 'GeneralLength': nan, 'GeneralWidth': nan, 'SeaFairingDepth': nan, 'PushedLength': nan, 'PushedWidth': nan, 'GeneralHeight': nan, 'SeaFairingLength': nan, 'SeaFairingWidth': nan, 'CoupledLength': nan, 'CoupledWidth': nan, 'PushedDepth': nan, 'WidePushedDepth': nan, 'CoupledDepth': nan, 'WidePushedLength': nan, 'WidePushedWidth': nan, 'SeaFairingHeight': nan, 'Id_navigability': nan, 'Classification': nan, 'Code': nan, 'Description': nan, 'length_deg': nan, 'length': 0.0255813258548747, 'Wkt': 'LINESTRING (3.54535894046351 51.727661900382, 3.5602008295784 51.742791566037, 3.56379034355024 51.7453141508194)', 'StartJunctionId': '8861095', 'EndJunctionId': '8864054', 'subgraph': 0, 'length_m': 2835.225880186762, 'geometry': <LINESTRING (3.545 51.728, 3.56 51.743, 3.564 51.745)>}


In [23]:
# PRINT FIS WATERDEPTH INFO 

# Create a map (pick a central point from your network)
m = folium.Map(location=[51.83, 4.33], zoom_start=10, tiles="cartodb positron")

for u, v, data in FG_FIS.edges(data=True):
    edge_geom = data["geometry"]  # This is already a Shapely LineString

    # Skip edges outside the polygon
    if not polygon.intersects(edge_geom):
        continue

    # Extract coordinates 
    points_x = list(data["geometry"].coords.xy[0])
    points_y = list(data["geometry"].coords.xy[1])
    line = [(points_y[i], points_x[i]) for i in range(len(points_x))]

    # Get depth value 
    # FIS
    depth = data.get("GeneralDepth", None)
    # EuRIS
    # depth_cm = data.get("mdraughtcm", None)

    # Convert to meters
    # if depth_cm is None or (isinstance(depth_cm, float) and np.isnan(depth_cm)):
    #     depth = None
    # else:
    #     depth = depth_cm / 100.0


    # Choose color based on depth
    if depth is None or (isinstance(depth, float) and np.isnan(depth)):
        color = "gray"
    else:
        if depth < 3:
            color = "red"
        elif depth < 6:
            color = "orange"
        else:
            color = "blue"

    # Add to map
    folium.PolyLine(
        line,
        color=color,
        weight=3,
        popup=f"Depth: {depth}"
    ).add_to(m)

# Display map
m

from folium.features import DivIcon

# # Noord label
# folium.Marker(
#     location=[51.85739439781898, 4.675508220128293],   # adjust slightly if needed
#     icon=DivIcon(
#         icon_size=(150,36),
#         icon_anchor=(0,0),
#         html='<div style="font-size:16px; font-weight:bold;">Noord <br> 4.4m</div>',
#     )
# ).add_to(m)

# # Scheldt–Rhine canal label
# folium.Marker(
#     location=[51.53096149571821, 4.236329052734132],   # adjust for best placement
#     icon=DivIcon(
#         icon_size=(250,36),
#         icon_anchor=(0,0),
#         html='<div style="font-size:16px; font-weight:bold;">Scheldt–Rhine canal <br> 4.3m</div>',
#     )
# ).add_to(m)



depth_noord = 4.4       
depth_scheldt_rhine = 4.3

# Noord depth label
folium.Marker(
    location=[51.87563442461901, 4.6579625210863105],  # adjust slightly if needed
    icon=DivIcon(
        icon_size=(60, 36),
        icon_anchor=(0, 0),
        html=f"""
        <div style="
            font-size:14px;
            font-weight:600;
            color:#000;
            background-color: rgba(255, 193, 7, 0.5);
            padding: 2px 2px;
            border-radius: 4px;
        ">
        Noord:<br> {depth_noord:.1f} m
        </div>
        """
    )
).add_to(m)

# Scheldt–Rhine canal depth label
folium.Marker(
    location=[51.53096149571821, 4.236329052734132],
    icon=DivIcon(
        icon_size=(120, 36),
        icon_anchor=(0, 0),
        html=f"""
        <div style="
            font-size:14px;
            font-weight:600;
            color:#000;
            background-color: rgba(255, 193, 7, 0.5);
            padding: 2px 2px;
            border-radius: 4px;
        ">
        Scheldt–Rhine:<br> {depth_scheldt_rhine:.1f} m
        </div>
        """
    )
).add_to(m)



from branca.element import Template, MacroElement

legend_html = """
{% macro html(this, kwargs) %}

<div style="
    position: fixed;
    bottom: 30px;
    left: 150px;
    width: 170px;
    background-color: white;
    border: 2px solid grey;
    z-index: 9999;
    font-size: 14px;
    padding: 10px;
">
<b>Water depth (m)</b><br>
<i style="background: red; width: 12px; height: 12px; float: left; margin-right: 6px; opacity: 0.9"></i>
&lt; 3 m<br>
<i style="background: orange; width: 12px; height: 12px; float: left; margin-right: 6px; opacity: 0.9"></i>
3 – 6 m<br>
<i style="background: blue; width: 12px; height: 12px; float: left; margin-right: 6px; opacity: 0.9"></i>
&gt; 6 m<br>
<i style="background: gray; width: 12px; height: 12px; float: left; margin-right: 6px; opacity: 0.9"></i>
No data
</div>

{% endmacro %}
"""

legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)


In [24]:
# PRINT FIS WATERDEPTH INFO 

# Create a map (pick a central point from your network)
m = folium.Map(location=[51.83, 4.33], zoom_start=10, tiles="cartodb positron")

for u, v, data in FG_EuRIS.edges(data=True):
    edge_geom = data["geometry"]  # This is already a Shapely LineString

    # Skip edges outside the polygon
    if not polygon.intersects(edge_geom):
        continue

    # Extract coordinates 
    points_x = list(data["geometry"].coords.xy[0])
    points_y = list(data["geometry"].coords.xy[1])
    line = [(points_y[i], points_x[i]) for i in range(len(points_x))]

    # Get depth value 
    # FIS
    # depth = data.get("GeneralDepth", None)
    # EuRIS
    depth_cm = data.get("mdraughtcm", None)

    # Convert to meters
    # if depth_cm is None or (isinstance(depth_cm, float) and np.isnan(depth_cm)):
    #     depth = None
    # else:
    #     depth = depth_cm / 100.0


    # Choose color based on depth
    if depth_cm is None or (isinstance(depth, float) and np.isnan(depth)):
        color = "gray"
    else:
        if depth_cm < 300:
            color = "red"
        elif depth_cm < 600:
            color = "orange"
        else:
            color = "blue"

    # Add to map
    folium.PolyLine(
        line,
        color=color,
        weight=3,
        popup=f"Depth: {depth_cm}"
    ).add_to(m)

# Display map
m